# GLIDE — Quickstart

**GLIDE** turns a small set of true evaluation labels (e.g. human annotations) plus and a large pool of proxy evaluation labels (e.g. LLM-as-judge annotations) into a **bias-corrected metric with a valid confidence interval**.

| Ingredient | Role |
|-----------|------|
| True labels | Ground truth — accurate but slow and expensive |
| Proxy labels | Proxy — cheap and fast but biased |

In [1]:
from glide.core.simulated import generate_dataset_binary
from glide.estimators.ppi import PPIMeanEstimator

## Step 1 — Assemble Your Data

A `Dataset` is a list of records, each record being a Python dict. Records come in two kinds:

- **Labeled** — have both `y_true` and `y_proxy`
- **Unlabeled** — have only `y_proxy`

Any key names work, pass them to `estimate()` at call time.

To run this notebook end-to-end, generate a synthetic dataset with the built-in helper:

We simulate 100 groundtruth-labeled plus 900 proxy-labeled conversations. 

The true rate is 15% and the proxy rate is 10% (biased low).

In [2]:
dataset = generate_dataset_binary(
    n=100,            # labeled records
    N=900,            # unlabeled records
    true_mean=0.15,   # true rate
    proxy_mean=0.10,  # proxy rate (biased)
    correlation=0.70,
    random_seed=0,
)

print(f"Dataset : {len(dataset)} records")
print()
print(f"Labeled sample   -> {dataset[0]}")
print(f"Unlabeled sample -> {dataset[-1]}")

Dataset : 1000 records

Labeled sample   -> {'y_true': 0, 'y_proxy': 0}
Unlabeled sample -> {'y_proxy': 0}


## Step 2 — Estimate the Metric

`PPIMeanEstimator.estimate()` corrects the proxy's bias using the labeled subset, then applies the correction across the full dataset.

When calling `estimate()`, we specify which columns of the dataset correspond to the true and proxy labels using the `y_true_field` and `y_proxy_field` parameters.

In [3]:
result = PPIMeanEstimator().estimate(
    dataset,
    y_true_field="y_true",
    y_proxy_field="y_proxy",
    metric_name="Evaluated metric", # e.g. a hallucination rate
    confidence_level=0.95,
)

print(result.summary())

Metric: Evaluated metric
Point Estimate: 0.138
Confidence Interval (95%): [0.08, 0.20]
Estimator : PPIMeanEstimator
n_true: 100
n_proxy: 1000
Effective Sample Size: 181.0


The effective sample size is the number of true labels needed to achieve the same confidence with true labels only.

## Step 3 — Read the Result

`InferenceResult` contains the point estimate, standard error and confidence interval.

In [4]:
# Point estimate and confidence interval
print(f"Estimate : {result.mean:.3}")
print(f"Standard error : {result.std:.3f}")
print(f"95% Confidence Interval : [{result.confidence_interval.lower_bound:.3}, \
{result.confidence_interval.upper_bound:.3}]")
print()

Estimate : 0.138
Standard error : 0.031
95% Confidence Interval : [0.0768, 0.2]



`InferenceResult` also allows you to perform hypothesis testing on the estimated distribution. For example, we can check if the 95% confidence interval is consistent with our target of keeping the rate below the specified limit.

In [5]:
# Test H0: hallucination rate = 10% (the judge's claim)
z, p, _ = result.confidence_interval.test_null_hypothesis(
    h0_value=0.10,
    alternative="larger",  # H1: true rate > 10%
)
print("H0: rate <= 10%  |  H1: rate > 10%")
print(f"z = {z:.2f}  |  p = {p:.4f}")
print("Reject H0 — the rate is significantly above 10%." 
      if p < 0.05 else "Cannot reject H0 at the 5% level.")

H0: rate <= 10%  |  H1: rate > 10%
z = 1.22  |  p = 0.1112
Cannot reject H0 at the 5% level.


Hypothesis was rejected, the hallucination rate is above the necessary limit of 10% at the 95% confidence level. In a real-world example, this would mean the system is not production-ready.

## Using Your Own Data Format

Replace `generate_dataset_binary` with any list of dicts. Labeled records must have both `y_true` and `y_proxy`; unlabeled records need only `y_proxy`.

```python
import json
import pandas as pd
import polars as pl

# From a JSON file
with open("annotations.json") as f:
    records = json.load(f)
dataset = Dataset(records)

# From a pandas DataFrame
df = pd.read_csv('your_data.csv')
records = df.to_dict(orient="records")
dataset = Dataset(records)

# From a polars DataFrame
df = pl.read_csv('your_data.csv')
records = df.to_dicts()
dataset = Dataset(records)